# Atelier Tensorflow
Contexte
Une entreprise dispose de données provenant de plusieurs bâtiments équipés de capteurs.
Pour chaque observation, on dispose par exemple de la température moyenne, de l’humidité, du
nombre d'occupants, de l’heure, du jour de la semaine, de la surface du bâtiment et de la
consommation énergétique.
L'objectif de l’atelier est de construire avec TensorFlow/Keras un réseau de neurones capable de
prédire la consommation énergétique d'un bâtiment à partir de caractéristiques telles que
température, humidité et nombre d'occupants. 

# 
2) Créer le notebook atelier_tensorflow_iot.ipynb
3) Installer et importer tensorflow, matplotlib et numpy

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# Partie 1 – Génération du dataset
1) Générer aléatoirement 1000 valeurs pour chacune des variables suivantes :
a) temperature : valeurs qui suivent une loi normale avec une moyenne de 25 °C et un
écart-type de 4 °C.
b) humidite : valeurs réparties de façon uniforme entre 30 % et 80 %.
c) occupants : valeurs entières choisies entre 1 et 49 inclus.
2) Déterminer la variable consommation avec la formule suivante :
 la consommation de base (0 °C, pas d'humidité et pièce vide) est de 50
 chaque degré supplémentaire augmente la consommation de 5 unités
 chaque pourcentage d'humidité en plus ajoute 1,5 unité à la consommation
 chaque personne présente dans la pièce augmente la consommation de 4 unités
 dans la vraie vie, une formule mathématique parfaite n'existe pas. On ajoute donc une
petite variation aléatoire (moyenne de 0 et écart-type de 10) pour simuler des imprévus
ou d'autres facteurs non mesurés.
3) Rassembler les variables (temperature, humidite et occupants) dans la matrice des
caractéristiques (features) X de taille 1000x3 en convertissant éventuellement les données au
format (float32) optimisé pour les calculs
4) Créer la cible (target) y qui contiendra la variable consommation, au format float32

In [5]:

# seed : fixe le "hasard" pour que tu obtiennes toujours les mêmes valeurs
# à chaque exécution (reproductibilité), exactement comme en NumPy
np.random.seed(42)

# ===== Génération des 3 caractéristiques (X) =====

# TEMPÉRATURE : loi normale
# loc=25    -> la MOYENNE de la distribution (le centre, où les valeurs se concentrent)
# scale=4   -> l'ÉCART-TYPE (la dispersion autour de la moyenne)
# size=1000 -> combien de valeurs générer
temperature = np.random.normal(loc=25, scale=4, size=1000).round(2)

# HUMIDITÉ : loi uniforme (aucune valeur n'est "plus probable" qu'une autre)
# low=30  -> la borne minimale possible
# high=80 -> la borne maximale possible
humidite = np.random.uniform(low=30, high=80, size=1000).round(2)

# OCCUPANTS : entiers aléatoires
# randint(1, 50) : tire un entier entre 1 et 49 INCLUS
# (rappel : randint EXCLUT toujours sa borne haute, donc on met 50 pour aller jusqu'à 49)
occupants = np.random.randint(1, 50, size=1000).round(2)

# ===== Calcul de la cible (y) =====

# Bruit aléatoire : simule les petites variations imprévisibles d'une vraie mesure
# loc=0    -> centré sur 0 (le bruit peut être positif OU négatif, en moyenne nul)
# scale=10 -> dispersion du bruit autour de 0
bruit = np.random.normal(loc=0, scale=10, size=1000)

# Formule donnée par l'énoncé : consommation de base (50) + effet de chaque variable
consommation = (50 + 5 * temperature + 1.5 * humidite + 4 * occupants + bruit).round(2)

# ===== Assemblage de X et y =====

# column_stack empile les 3 tableaux CÔTE À CÔTE, comme 3 colonnes d'un tableau Excel
# résultat : une matrice de forme (1000, 3) -> 1000 lignes, 3 colonnes
# .astype("float32") : conversion demandée, format optimisé pour TensorFlow
X = np.column_stack([temperature, humidite, occupants]).astype("float32")

# y : juste la liste des 1000 valeurs de consommation, au même format
y = consommation.astype("float32")

print("Shape de X :", X.shape)   # doit afficher (1000, 3)
print("Shape de y :", y.shape)   # doit afficher (1000,)
print("Premières lignes de X :\n", X[:5])
print("Premières valeurs de y :\n", y[:5])

Shape de X : (1000, 3)
Shape de y : (1000,)
Premières lignes de X :
 [[26.99 38.37  8.  ]
 [24.45 35.23 43.  ]
 [27.59 61.82 20.  ]
 [31.09 65.32 33.  ]
 [24.06 31.58 26.  ]]
Premières valeurs de y :
 [281.64 406.7  348.62 433.62 318.93]


# Partie 2 – Découpage Train/Test
Diviser le dataset précédent (X et y) en deux ensembles distincts : un pour l'entraînement (train)
et un pour le test (test). Avec les conditions suivantes : 20% des données serviront au test ; garantir
la reproductibilité du découpage. 

In [6]:
from sklearn.model_selection import train_test_split

# X_train, X_test : caractéristiques (température, humidité, occupants)
# y_train, y_test : cible (consommation)
# test_size=0.2   -> 20% des données pour le test (demandé par l'énoncé)
# random_state=42 -> garantit la reproductibilité (demandé par l'énoncé)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Taille X_train :", X_train.shape)
print("Taille X_test  :", X_test.shape)

Taille X_train : (800, 3)
Taille X_test  : (200, 3)


# Partie 3 – Création du modèle
1) Construire un réseau de neurones qui utilise l’architecture séquentielle suivante :
a) couche 1 : 16 neurones avec relu comme fonction d’activation
b) couche 2 : 8 neurones avec relu
c) couche 3 : 1 seul neurone.
2) Afficher un résumé textuel de l'architecture du réseau de neurones. 

In [7]:
# Architecture séquentielle : les données traversent les couches dans l'ordre, une seule voie
modele = tf.keras.Sequential([
    # Couche d'entrée : chaque exemple a 3 valeurs (température, humidité, occupants)
    tf.keras.layers.Input(shape=(3,)),
    
    # Couche 1 : 16 neurones, activation relu
    # (relu transforme toute valeur négative en 0, laisse passer les valeurs positives)
    tf.keras.layers.Dense(16, activation="relu"),
    
    # Couche 2 : 8 neurones, activation relu
    tf.keras.layers.Dense(8, activation="relu"),
    
    # Couche 3 (sortie) : 1 seul neurone, PAS d'activation
    # (on prédit un nombre continu -> la consommation -> c'est une régression,
    # donc pas besoin de "sigmoid" ou "softmax" comme pour une classification)
    tf.keras.layers.Dense(1)
])

# Affiche un résumé de l'architecture : nombre de couches, de paramètres, etc.
modele.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16)             │            64 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 209 (836.00 B)

 Trainable params: 209 (836.00 B)

 Non-trainable params: 0 (0.00 B)